# Get rings around object borders

This notebook takes exisiting integer label maps and extracts **rings (bands) of pixels at a given distance range from object borders**, either inside the objects, outside of them, or on both sides.

Distances are measured with the Euclidean distance transform (EDT):

- inside: distance to the nearest pixel not belonging to the same object: background, pixels of other objects (shared interfaces of touching objects, see `interfaces_as_border`), internal holes, or the image edge
- outside: distance to the nearest object pixel; background pixels are assigned the label of the closest object

The outermost pixel of an object and the first pixel outside of it both have distance 1.

## Required Inputs

- a folder containing input masks as integer TIFF files

## Outputs

- a new folder with ring label maps as TIFF files (same filenames as input files). Labels stay the same; pixels outside the ring are 0.

In [ ]:
from enum import StrEnum, auto

from rings import ring_labels


class Side(StrEnum):
    INSIDE = auto()
    OUTSIDE = auto()
    BOTH = auto()

In [ ]:
in_path = '/home/david/Desktop/symbiosis_analysis_guse/'

# subfolder containing input labels
labels_subdirectory = 'segmentation_symbionts'
# where to save results to
out_subdirectory = 'segmentation_ring'

# where to take the ring relative to the border: 'inside', 'outside' or 'both'
side = 'both'

# distance range (pixels) from the border covered by the ring: (start, end)
# (border-adjacent pixels have distance 1)
distances = (0, 5)

# whether shared interfaces of touching objects count towards the object border
# (True, matching per-object erosion) or only background does (False)
interfaces_as_border = True

In [ ]:
from pathlib import Path
from natsort import natsorted

# check that we selected a valid side
if side not in Side:
    raise ValueError(f'Undefined side {side}, available options: {set(Side)}')

# unpack and check distance range
start, end = distances
if end < max(start, 1):
    raise ValueError(f'end distance ({end}) must be >= start distance ({start})')

in_files = natsorted((Path(in_path) / labels_subdirectory).glob('[!.]*.tif'))
in_files

In [ ]:
from skimage.io import imread
from tifffile import imwrite

# make out directory if it doesn't exist already
out_path = Path(in_path) / out_subdirectory
if not out_path.exists():
    out_path.mkdir()

for in_file in in_files:

    # load
    labels = imread(in_file)

    # extract ring at requested distances
    labels_modified = ring_labels(
        labels,
        start=start,
        end=end,
        side=side,
        interfaces_as_border=interfaces_as_border,
    )

    # save to output folder with same name
    out_file = out_path / in_file.name
    imwrite(out_file, labels_modified, compression=5)
    
    print(f'{in_file} done.')

## TESTING code below -> commented out for papermill usage

In [ ]:
# from matplotlib import pyplot as plt
# from skimage.io import imread

# from rings import ring_labels

# labels = imread('test_labels.tif')
# labels_inside = ring_labels(labels, 0, 7, side='inside')
# labels_outside = ring_labels(labels, 0, 7, side='outside')

# fig, axs = plt.subplots(ncols=3, figsize=(12,4))
# axs[0].imshow(labels)
# axs[1].imshow(labels_inside)
# axs[2].imshow(labels_outside)

In [ ]:
# import napari

# if napari.current_viewer() is not None:
#     napari.current_viewer().close()

# viewer = napari.Viewer()

# viewer.add_labels(ring_labels(labels, 0, 7, side='both'))